# Oracle-RC Evaluation Table Generator

This notebook computes Oracle-RC (Response-Calibrated) baseline metrics across multiple models and datasets.

Oracle-RC randomly selects one sample's correctness (0 or 1) as the confidence for each question. We average results across 5 random seeds to reduce variance.

In [1]:
import json
import sys
from pathlib import Path
from typing import Dict, List, Tuple

import numpy as np
import pandas as pd

# Add src to path for imports
sys.path.insert(0, str(Path.cwd().parent))
from src.metrics.custom import c_star_metrics

In [2]:
# ===== CONFIGURATION =====

# Models to evaluate (choose which models to include)
models = [
    'Olmo-3-7B-Instruct',
    'Qwen3-8B-non-thinking',
    'gpt-oss-20b',
]

# Datasets to evaluate (internal names)
datasets = [
    'triviaqa-validation',
    'simpleqa-verified',
    'gsm8k-test',
    'math-500',
    'aime25-test',
    'mmlu-validation',
    'gpqa-diamond',
]

# Dataset display name mapping
dataset_display_names = {
    'triviaqa-validation': 'TriviaQA',
    'simpleqa-verified': 'SimpleQA',
    'gsm8k-test': 'GSM8K',
    'math-500': 'MATH',
    'aime25-test': 'AIME25',
    'mmlu-validation': 'MMLU',
    'gpqa-diamond': 'GPQA',
}

# Metrics to compute (all 4 available)
metrics_to_compute = ['mse', 'spearman_r'] # 'ece', 'pearson_r'

# Paths
outputs_dir = Path('../outputs')

# Oracle-RC parameters
num_seeds = 5  # Number of random seeds to average over
base_seed = 42  # Starting seed

In [3]:
# ===== HELPER FUNCTIONS =====

def find_output_folder(outputs_dir: Path, model: str, dataset: str) -> Path:
    """
    Find the output folder for a given model and dataset.
    Format: {dataset}__{model}
    """
    folder_name = f"{dataset}__{model}"
    folder_path = outputs_dir / folder_name
    
    if not folder_path.exists():
        raise FileNotFoundError(f"No folder found: {folder_path}")
    
    return folder_path


def load_ground_truth(folder_path: Path) -> Tuple[Dict[str, float], Dict[str, List[int]], List[str]]:
    """
    Load ground truth data from ground_truth.jsonl.
    
    Returns:
        - expected_accuracy_map: Dict[example_id -> expected_accuracy]
        - samples_map: Dict[example_id -> List[0/1 correctness values]]
        - example_ids: Sorted list of example IDs
    """
    gt_path = folder_path / "ground_truth.jsonl"
    
    if not gt_path.exists():
        raise FileNotFoundError(f"ground_truth.jsonl not found in {folder_path}")
    
    expected_accuracy_map = {}
    samples_map = {}
    
    with open(gt_path, 'r') as f:
        for line in f:
            data = json.loads(line)
            example_id = data['example_id']
            expected_accuracy = data['expected_accuracy']
            num_samples = data['num_samples']
            num_correct = data['num_correct']
            
            expected_accuracy_map[example_id] = expected_accuracy
            
            # Reconstruct sample list: [1, 1, ..., 1, 0, 0, ..., 0]
            sample_list = [1] * num_correct + [0] * (num_samples - num_correct)
            samples_map[example_id] = sample_list
    
    example_ids = sorted(expected_accuracy_map.keys())
    
    return expected_accuracy_map, samples_map, example_ids


def compute_oracle_rc_confidence(samples_map: Dict[str, List[int]], 
                                  example_ids: List[str], 
                                  seed: int) -> List[float]:
    """
    Compute Oracle-RC confidence: randomly select one sample's correctness per question.
    
    Returns:
        List of confidence values (0.0 or 1.0) in order of example_ids
    """
    rng = np.random.RandomState(seed)
    confidences = []
    
    for eid in example_ids:
        sample_list = samples_map[eid]
        random_idx = rng.randint(0, len(sample_list))
        confidence = float(sample_list[random_idx])
        confidences.append(confidence)
    
    return confidences

In [4]:
def compute_oracle_rc_metrics(model: str, dataset: str, 
                               num_seeds: int = 5, base_seed: int = 42) -> Dict[str, float]:
    """
    Compute Oracle-RC metrics for a model-dataset pair, averaged over multiple seeds.
    
    Returns:
        Dict with keys: 'mse', 'ece', 'pearson_r', 'spearman_r'
    """
    try:
        # Find output folder
        folder = find_output_folder(outputs_dir, model, dataset)
        
        # Load ground truth
        expected_acc_map, samples_map, example_ids = load_ground_truth(folder)
        
        # Get ground truth expected accuracies in order
        c_star = [expected_acc_map[eid] for eid in example_ids]
        
        # Compute metrics for each seed
        all_metrics = {
            'mse': [],
            'ece': [],
            'pearson_r': [],
            'spearman_r': [],
        }
        
        for seed_idx in range(num_seeds):
            seed = base_seed + seed_idx
            
            # Get Oracle-RC confidences (binary: 0.0 or 1.0)
            oracle_rc_conf = compute_oracle_rc_confidence(samples_map, example_ids, seed)
            
            # Compute metrics
            metrics = c_star_metrics(oracle_rc_conf, c_star)
            
            all_metrics['mse'].append(metrics.mse)
            all_metrics['ece'].append(metrics.ece)
            all_metrics['pearson_r'].append(metrics.pearson_r)
            all_metrics['spearman_r'].append(metrics.spearman_r)
        
        # Average across seeds
        avg_metrics = {
            metric_name: np.mean(values)
            for metric_name, values in all_metrics.items()
        }
        
        return avg_metrics
        
    except FileNotFoundError as e:
        print(f"  ⚠️  Skipping {model} / {dataset}: {e}")
        return None
    except Exception as e:
        print(f"  ❌ Error processing {model} / {dataset}: {e}")
        return None

In [5]:
# ===== COMPUTE ORACLE-RC METRICS =====

print("Computing Oracle-RC metrics...")
print(f"Models: {models}")
print(f"Datasets: {datasets}")
print(f"Averaging over {num_seeds} seeds\n")

# Store results: results[model][dataset][metric] = value
results = {}

for model in models:
    print(f"\n{'='*60}")
    print(f"Model: {model}")
    print(f"{'='*60}")
    
    results[model] = {}
    
    for dataset in datasets:
        print(f"  Processing {dataset}...", end=' ')
        
        metrics = compute_oracle_rc_metrics(model, dataset, num_seeds, base_seed)
        
        if metrics is not None:
            results[model][dataset] = metrics
            print(f"✓ (MSE={metrics['mse']:.4f}, ECE={metrics['ece']:.4f})")
        else:
            results[model][dataset] = None

print(f"\n{'='*60}")
print("Computation complete!")
print(f"{'='*60}")

Computing Oracle-RC metrics...
Models: ['Olmo-3-7B-Instruct', 'Qwen3-8B-non-thinking', 'gpt-oss-20b']
Datasets: ['triviaqa-validation', 'simpleqa-verified', 'gsm8k-test', 'math-500', 'aime25-test', 'mmlu-validation', 'gpqa-diamond']
Averaging over 5 seeds


Model: Olmo-3-7B-Instruct
  Processing triviaqa-validation... ✓ (MSE=0.0710, ECE=0.1401)
  Processing simpleqa-verified... ✓ (MSE=0.0181, ECE=0.0371)
  Processing gsm8k-test... ✓ (MSE=0.0239, ECE=0.0465)
  Processing math-500... ✓ (MSE=0.0346, ECE=0.0713)
  Processing aime25-test... ✓ (MSE=0.0815, ECE=0.1748)
  Processing mmlu-validation... ✓ (MSE=0.0803, ECE=0.1589)
  Processing gpqa-diamond... ✓ (MSE=0.1175, ECE=0.2419)

Model: Qwen3-8B-non-thinking
  Processing triviaqa-validation... ✓ (MSE=0.0531, ECE=0.1063)
  Processing simpleqa-verified... ✓ (MSE=0.0203, ECE=0.0428)
  Processing gsm8k-test... ✓ (MSE=0.0186, ECE=0.0376)
  Processing math-500... ✓ (MSE=0.0588, ECE=0.1140)
  Processing aime25-test... ✓ (MSE=0.0534, ECE=0.1101)
 

In [6]:
# ===== GENERATE TABLES =====

def create_metric_table(results: Dict, models: List[str], datasets: List[str], 
                        metric_name: str) -> pd.DataFrame:
    """
    Create a DataFrame table for a specific metric.
    
    Rows: Models
    Columns: Datasets (with display names)
    Values: Metric values
    """
    # Create column names with display names
    column_names = [dataset_display_names.get(ds, ds) for ds in datasets]
    
    # Build data matrix
    data = []
    row_names = []
    
    for model in models:
        row = []
        for dataset in datasets:
            if model in results and dataset in results[model] and results[model][dataset] is not None:
                value = results[model][dataset][metric_name]
                row.append(value)
            else:
                row.append(np.nan)
        
        data.append(row)
        row_names.append(model)
    
    # Create DataFrame
    df = pd.DataFrame(data, index=row_names, columns=column_names)
    
    return df


# Create one table per metric
tables = {}

for metric in metrics_to_compute:
    tables[metric] = create_metric_table(results, models, datasets, metric)

print("Tables created for metrics:", list(tables.keys()))

Tables created for metrics: ['mse', 'spearman_r']


In [7]:
# ===== DISPLAY TABLES (EXCEL-FRIENDLY) =====

print("\n" + "="*80)
print("ORACLE-RC EVALUATION RESULTS")
print("="*80)
print("\nTo copy to Excel: Select the displayed table and copy-paste\n")

for metric in metrics_to_compute:
    print(f"\n{'='*80}")
    print(f"Metric: {metric.upper()}")
    print(f"{'='*80}\n")
    
    df = tables[metric].round(4)
    
    # Display DataFrame (in Jupyter, this creates a nice HTML table that can be copied to Excel)
    display(df)
    
    # Also print as tab-separated for easy copy-paste
    print("\nTab-separated format (can copy-paste to Excel):")
    print(df.to_csv(sep='\t', na_rep='N/A'))
    print()


ORACLE-RC EVALUATION RESULTS

To copy to Excel: Select the displayed table and copy-paste


Metric: MSE



,TriviaQA,SimpleQA,GSM8K,MATH,AIME25,MMLU,GPQA
Olmo-3-7B-Instruct,0.0710,0.0181,0.0239,0.0346,0.0815,0.0803,0.1175
Qwen3-8B-non-thinking,0.0531,0.0203,0.0186,0.0588,0.0534,0.0507,0.1332
gpt-oss-20b,0.0672,0.0322,0.0160,0.0172,0.0972,0.0297,0.0923



Tab-separated format (can copy-paste to Excel):
	TriviaQA	SimpleQA	GSM8K	MATH	AIME25	MMLU	GPQA
Olmo-3-7B-Instruct	0.071	0.0181	0.0239	0.0346	0.0815	0.0803	0.1175
Qwen3-8B-non-thinking	0.0531	0.0203	0.0186	0.0588	0.0534	0.0507	0.1332
gpt-oss-20b	0.0672	0.0322	0.016	0.0172	0.0972	0.0297	0.0923



Metric: SPEARMAN_R



,TriviaQA,SimpleQA,GSM8K,MATH,AIME25,MMLU,GPQA
Olmo-3-7B-Instruct,0.8154,0.3723,0.5675,0.5362,0.7826,0.7243,0.7018
Qwen3-8B-non-thinking,0.8287,0.4620,0.5863,0.6495,0.7281,0.7196,0.6815
gpt-oss-20b,0.7792,0.4077,0.4682,0.3520,0.6353,0.5649,0.7244



Tab-separated format (can copy-paste to Excel):
	TriviaQA	SimpleQA	GSM8K	MATH	AIME25	MMLU	GPQA
Olmo-3-7B-Instruct	0.8154	0.3723	0.5675	0.5362	0.7826	0.7243	0.7018
Qwen3-8B-non-thinking	0.8287	0.462	0.5863	0.6495	0.7281	0.7196	0.6815
gpt-oss-20b	0.7792	0.4077	0.4682	0.352	0.6353	0.5649	0.7244


